#### 03. Эксперименты

Здесь проверяем гипотезы из EDA: новые признаки, preprocessing и модели.

In [1]:
import torch
import pandas as pd

from hydra.utils import instantiate
from omegaconf import OmegaConf

from titanic.data import load
from titanic.evaluate import evaluate, run_grid_search
from titanic.paths import CONFIG_DIR
from titanic.pipelines import build_pipeline, build_catboost_pipeline, build_neural_pipeline

In [2]:
df_train, df_test = load()

X = df_train.drop(columns='Survived')
y = df_train.Survived

print('Размер train:', df_train.shape)
print('Размер test:', df_test.shape)

Размер train: (891, 12)
Размер test: (418, 11)


#### Эксперимент 1: Logistic Regression + новые признаки

Проверим стабильную конфигурацию Logistic Regression на общем наборе признаков.

К исходным признакам добавлены `Title`, `FamilyGroup`, `Deck`, `IsChild` и `AgeMissing`. Ниже сохранён поиск, с помощью которого была выбрана сила регуляризации.

In [12]:
# Загружаем стабильную конфигурацию Logistic Regression.
config = OmegaConf.load(CONFIG_DIR/'logistic_regression'/'01_stable.yaml')

# Hydra создает модель по секции model.
model = instantiate(config.model)

pipeline = build_pipeline(model)

logistic_scores = evaluate(
    pipeline=pipeline,
    X=X,
    y=y
)

Accuracy по фолдам: 0.844, 0.826, 0.820, 0.837, 0.831
Средняя accuracy: 0.832
Std accuracy: 0.008


Стабильная Logistic Regression получила accuracy `0.832` — на `0.037` выше raw baseline. Разброс между фолдами небольшой (`std = 0.008`).

Новый набор признаков оказался полезным, а более сильная регуляризация сделала результат стабильнее.

**Итерация 2: подбор силы регуляризации**

Поиск начинался со стандартного значения `C=1.0`.

Теперь проверим несколько значений `C`. Чем меньше `C`, тем сильнее модель ограничивает свои коэффициенты. Чем больше `C`, тем слабее регуляризация и тем свободнее модель подстраивается под обучающие данные.

Признаки, остальные параметры и разбиение на фолды не меняем.

In [13]:
# Загружаем параметры модели и значения C для проверки
config = OmegaConf.load(CONFIG_DIR/'logistic_regression'/'02_regularization.yaml')

# Hydra создает модель по секции model.
model = instantiate(config.model)

pipeline = build_pipeline(model)

# Передаем в GridSearch только 
# параметры, которые хотим подобрать.
param_grid = {
    'logisticregression__C': list(config.search.C)
}

# Проверяем все значения C на общей схеме кросс-валидации.
logistic_search = run_grid_search(
    pipeline=pipeline,
    param_grid=param_grid,
    X=X,
    y=y
)

Сравним качество на обучающих и валидационных фолдах.

Если `Train accuracy` растёт, а `CV accuracy` не меняется или падает, ослабление регуляризации уже не помогает модели лучше работать на новых данных.

In [14]:
result_columns = {
    'param_logisticregression__C': 'C',
    'mean_train_scores': 'Train accuracy',
    'mean_test_score': 'CV accuracy',
    'std_test_score': 'CV std'
}

logistic_search_results = (
    pd.DataFrame(logistic_search.cv_results_)
    .rename(columns=result_columns)
    .filter(items=tuple(result_columns.values()))
    .astype({'C': 'float64'})
    .sort_values('C')
    .reset_index(drop=True)
)

display(
    logistic_search_results.style
    .hide(axis='index')
    .format({
        'C': '{:g}',
        'Train accuracy': '{:.3f}',
        'CV accuracy': '{:.3f}',
        'CV std': '{:.3f}',
    })
)

print('Лучшее C', logistic_search.best_params_['logisticregression__C'])
print(f'Лучшая CV accuracy: {logistic_search.best_score_:.3f}')

C,CV accuracy,CV std
0.01,0.804,0.013
0.1,0.832,0.008
1,0.832,0.015
10,0.827,0.017
100,0.826,0.014


Лучшее C 0.1
Лучшая CV accuracy: 0.832


Вывод

Лучший результат получен при `C=0.1`: средняя accuracy осталась на уровне `0.832`, но разброс между фолдами снизился с `0.015` до `0.008`.

Слишком сильная регуляризация при `C=0.01` ухудшает качество, а значения выше `1.0` не дают прироста. Для Logistic Regression фиксируем `C=0.1`.

#### Эксперимент 2: Support Vector Machine

Logistic Regression строит линейную границу между классами. Теперь проверим SVC с RBF ядром, который способен находить нелинейные зависимости между признаками.

Сначала проверим стабильную конфигурацию на том же наборе признаков и тех же CV-фолдах. Ниже сохранены обе итерации подбора `C` и `gamma`.

In [15]:
# Загружаем стабильную конфигурацию SVC.
config = OmegaConf.load(CONFIG_DIR/'svc'/'01_stable.yaml')

# Hydra создает модель по секции model.
model = instantiate(config.model)

pipeline = build_pipeline(model)

svc_scores = evaluate(
    pipeline=pipeline,
    X=X,
    y=y
)

Accuracy по фолдам: 0.844, 0.826, 0.826, 0.831, 0.848
Средняя accuracy: 0.835
Std accuracy: 0.009


Стабильный SVC получил accuracy `0.835` со значением `std = 0.009`. Это на `0.003` выше Logistic Regression.

Модель немного точнее Logistic Regression, но преимущество слишком мало, чтобы считать его однозначным.

**Итерация 2: подбор `C` и `gamma`**

Теперь подберём два основных параметра SVC с RBF-ядром.

`C` управляет силой регуляризации: маленькое значение сильнее ограничивает модель, большое позволяет ей точнее подстраиваться под обучающие данные.

`gamma` определяет область влияния отдельных наблюдений. При маленьком значении граница получается более плавной, при большом — более сложной и локальной.

Признаки, метрику и разбиение на фолды не меняем.

In [ ]:
# Загружаем параметры SVC и сетку для поиска.
config = OmegaConf.load(CONFIG_DIR/'svc'/'02_grid_search.yaml')

# Hydra создает модель по секции model.
model = instantiate(config.model)

pipeline = build_pipeline(model)

# Передаём GridSearchCV значения C и gamma.
param_grid = {
    'svc__C': list(config.search.C),
    'svc__gamma': list(config.search.gamma),
}

# Проверяем все комбинации на общей схеме кросс-валидации.
svc_search = run_grid_search(
    pipeline=pipeline,
    param_grid=param_grid,
    X=X,
    y=y,
)

Сравним все комбинации параметров. В таблице также оставим качество на train, чтобы увидеть варианты, которые начинают переобучаться.

In [17]:
result_columns = {
    'rank_test_score': 'Rank',
    'param_svc__C': 'C',
    'param_svc__gamma': 'Gamma',
    'mean_train_score': 'Train accuracy',
    'mean_test_score': 'CV accuracy',
    'std_test_score': 'CV std'
}

svc_search_results = (
    pd.DataFrame(svc_search.cv_results_)
    .rename(columns=result_columns)
    .filter(items=tuple(result_columns.values()))
    .astype({'Rank': 'int64', 'C': 'float64'})
    .sort_values('Rank')
    .reset_index(drop=True)
)

display(
    svc_search_results.style
    .hide(axis='index')
    .format({
        'C': '{:g}',
        'Train accuracy': '{:.3f}',
        'CV accuracy': '{:.3f}',
        'CV std': '{:.3f}'
    })
)

print('Лучшее C:', svc_search.best_params_['svc__C'])
print('Лучшая gamma:', svc_search.best_params_['svc__gamma'])
print(f'Лучшая CV accuracy: {svc_search.best_score_:.3f}')

Rank,C,Gamma,Train accuracy,CV accuracy,CV std
1,10,0.010000,0.835,0.835,0.009
2,0.1,0.100000,0.829,0.831,0.013
3,0.1,scale,0.829,0.829,0.012
4,1,0.100000,0.842,0.829,0.017
4,1,scale,0.840,0.829,0.017
6,1,0.010000,0.831,0.825,0.013
7,100,0.010000,0.859,0.820,0.019
8,10,0.100000,0.894,0.818,0.026
9,10,scale,0.892,0.816,0.027
10,100,0.100000,0.912,0.808,0.027


Лучшее C: 10.0
Лучшая gamma: 0.01
Лучшая CV accuracy: 0.835


Вывод

Лучший результат получен при `C=10` и `gamma=0.01`: CV accuracy выросла с `0.829` до `0.835`, а разброс снизился до `0.009`.

У лучшей модели нет заметного разрыва между train и CV. При больших `C` и `gamma` train accuracy продолжает расти, а CV accuracy падает — модель начинает переобучаться.

Поскольку лучшее `gamma=0.01` находится на границе проверенного диапазона, проведём ещё один поиск около найденных значений.

**Итерация 3: уточнение `C` и `gamma`**

Лучшее значение `gamma=0.01` оказалось на границе предыдущего диапазона. Расширим поиск в сторону меньших значений и точнее проверим область около `C=10`.

In [18]:
# Загружаем параметры SVC и сетку для поиска.
config = OmegaConf.load(CONFIG_DIR/'svc'/'03_refined_search.yaml')

# Hydra создает модель по секции model.
model = instantiate(config.model)

pipeline = build_pipeline(model)

# Передаём GridSearchCV значения C и gamma.
param_grid = {
    'svc__C': list(config.search.C),
    'svc__gamma': list(config.search.gamma),
}

# Проверяем все комбинации на общей схеме кросс-валидации.
svc_refined_search = run_grid_search(
    pipeline=pipeline,
    param_grid=param_grid,
    X=X,
    y=y,
)

In [19]:
result_columns = {
    'rank_test_score': 'Rank',
    'param_svc__C': 'C',
    'param_svc__gamma': 'Gamma',
    'mean_train_score': 'Train accuracy',
    'mean_test_score': 'CV accuracy',
    'std_test_score': 'CV std'
}

svc_search_results = (
    pd.DataFrame(svc_refined_search.cv_results_)
    .rename(columns=result_columns)
    .filter(items=tuple(result_columns.values()))
    .astype({'Rank': 'int64', 'C': 'float64'})
    .sort_values('Rank')
    .reset_index(drop=True)
)

display(
    svc_search_results.style
    .hide(axis='index')
    .format({
        'C': '{:g}',
        'Train accuracy': '{:.3f}',
        'CV accuracy': '{:.3f}',
        'CV std': '{:.3f}'
    })
)

print('Лучшее C:', svc_refined_search.best_params_['svc__C'])
print('Лучшая gamma:', svc_refined_search.best_params_['svc__gamma'])
print(f'Лучшая CV accuracy: {svc_refined_search.best_score_:.3f}')

Rank,C,Gamma,Train accuracy,CV accuracy,CV std
1,10,0.010000,0.835,0.835,0.009
1,10,0.003000,0.835,0.835,0.009
1,10,0.005000,0.835,0.835,0.009
1,20,0.003000,0.835,0.835,0.009
1,30,0.003000,0.835,0.835,0.009
1,30,0.005000,0.835,0.835,0.009
1,20,0.005000,0.835,0.835,0.009
8,5,0.005000,0.834,0.834,0.010
8,20,0.010000,0.836,0.834,0.010
10,5,0.030000,0.841,0.834,0.012


Лучшее C: 10.0
Лучшая gamma: 0.003
Лучшая CV accuracy: 0.835


Вывод

Лучшие комбинации находятся в области `C=10–30` и `gamma=0.003–0.01`. Они дают одинаковую CV accuracy `0.835` с разбросом `0.009` и без заметного разрыва между train и CV.

Дальнейшее увеличение `C` и `gamma` повышает качество на train, но ухудшает CV — начинается переобучение.

Среди равных вариантов фиксируем `C=10` и `gamma=0.003`: это наименее сложная комбинация из найденного устойчивого диапазона.

#### Эксперимент 3: Decision Tree

Теперь проверим одиночное дерево решений. Оно умеет самостоятельно находить нелинейные зависимости и взаимодействия признаков, но без ограничений может легко переобучиться.

На первом запуске оставляем дерево почти без ограничений, чтобы получить его базовый результат. Настройкой глубины и размера листьев займёмся отдельно, если модель окажется перспективной.

In [28]:
# Загружаем стабильную конфигурацию дерева решений.
config = OmegaConf.load(CONFIG_DIR/'decision_tree'/'01_stable.yaml')

# Создаем DecisionTreeClassifier из YAML.
model = instantiate(config.model)

pipeline = build_pipeline(model)

decision_tree_scores = evaluate(
    pipeline=pipeline,
    X=X,
    y=y
)

Accuracy по фолдам: 0.804, 0.815, 0.730, 0.809, 0.792
Средняя accuracy: 0.790
Std accuracy: 0.031


Дерево без ограничений получило accuracy 0.790 — хуже даже baseline с результатом 0.795 и на 0.042 хуже текущего лидера.

Разброс между фолдами высокий (`std = 0.031`), а на одном из них accuracy упала до 0.730. Значит, одиночное дерево сильно зависит от конкретных обучающих данных и в текущем виде работает нестабильно.

По одной CV нельзя строго доказать переобучение, но для полностью выросшего дерева это наиболее вероятная причина. Позже можно проверить ограничения `max_depth` и `min_samples_leaf`, а сейчас логично перейти к Random Forest, который как раз уменьшает нестабильность одиночных деревьев.

#### Эксперимент 4: Random Forest

Одиночное дерево может сильно зависеть от конкретной обучающей выборки. Random Forest обучает множество разных деревьев и объединяет их ответы, благодаря чему результат обычно получается устойчивее.

Сначала проверим стабильную конфигурацию с 500 деревьями. Ниже сохранён поиск глубины, размера листьев и числа признаков.

In [21]:
# Загружаем стабильную конфигурацию Random Forest.
config = OmegaConf.load(CONFIG_DIR/'random_forest'/'01_stable.yaml')

# Создаём RandomForestClassifier из YAML.
model = instantiate(config.model)

pipeline = build_pipeline(model)

random_forest_scores = evaluate(
    pipeline=pipeline,
    X=X,
    y=y,
)

Accuracy по фолдам: 0.866, 0.860, 0.809, 0.848, 0.843
Средняя accuracy: 0.845
Std accuracy: 0.020


Стабильный Random Forest получил accuracy `0.845` при `std = 0.020` и стал текущим лидером.

Он превосходит стабильный SVC на `0.010`, хотя сильнее зависит от конкретного разбиения данных.

**Итерация 2: ограничение сложности Random Forest**

Базовый Random Forest обучал деревья практически без ограничений. Теперь проверим глубину деревьев, минимальное число пассажиров в листе и число доступных признаков при разделении.

In [26]:
# Загружаем параметры Random Forest и сетку поиска.
config = OmegaConf.load(CONFIG_DIR/'random_forest'/'02_grid_search.yaml')

# Создаём RandomForestClassifier из YAML.
model = instantiate(config.model)

pipeline = build_pipeline(model)

# Передаём GridSearchCV параметры,
# которые управляют сложностью отдельных деревьев.
param_grid = {
    'randomforestclassifier__max_depth': list(
        config.search.max_depth
    ),
    'randomforestclassifier__min_samples_leaf': list(
        config.search.min_samples_leaf
    ),
    'randomforestclassifier__max_features': list(
        config.search.max_features
    ),
}

# Проверяем все комбинации на общей схеме кросс-валидации.
random_forest_search = run_grid_search(
    pipeline=pipeline,
    param_grid=param_grid,
    X=X,
    y=y,
)

Выведем лучшие комбинации параметров. Особое внимание уделим разнице между train и CV: большой разрыв будет указывать на переобучение леса.

In [27]:
result_columns = {
    'rank_test_score': 'Rank',
    'param_randomforestclassifier__max_depth': 'Max depth',
    'param_randomforestclassifier__min_samples_leaf': 'Min samples leaf',
    'param_randomforestclassifier__max_features': 'Max features',
    'mean_train_score': 'Train accuracy',
    'mean_test_score': 'CV accuracy',
    'std_test_score': 'CV std'
}

random_forest_search_results = (
    pd.DataFrame(random_forest_search.cv_results_)
    .rename(columns=result_columns)
    .filter(items=tuple(result_columns.values()))
    .sort_values(['Rank', 'CV std'])
    .reset_index(drop=True)
)

display(
    random_forest_search_results
    .head(15)
    .style
    .hide(axis='index')
    .format({
        'Train accuracy': '{:.3f}',
        'CV accuracy': '{:.3f}',
        'CV std': '{:.3f}'
    })
)

print('Лучшая глубина:', random_forest_search.best_params_['randomforestclassifier__max_depth'])
print('Лучший min_samples_leaf:', random_forest_search.best_params_['randomforestclassifier__min_samples_leaf'])
print('Лучший max_features:', random_forest_search.best_params_['randomforestclassifier__max_features'])
print(f'Лучшая CV accuracy: ' f'{random_forest_search.best_score_:.3f}')

Rank,Max depth,Min samples leaf,Max features,Train accuracy,CV accuracy,CV std
1,8,1,None,0.937,0.845,0.020
2,6,1,None,0.904,0.839,0.028
3,12,4,None,0.901,0.838,0.017
3,16,4,None,0.901,0.838,0.017
3,None,4,None,0.901,0.838,0.017
6,12,2,None,0.929,0.838,0.017
6,10,2,None,0.925,0.838,0.019
6,10,1,None,0.960,0.838,0.022
9,16,4,sqrt,0.867,0.837,0.015
10,8,1,0.500000,0.931,0.837,0.014


Лучшая глубина: 8
Лучший min_samples_leaf: 1
Лучший max_features: None
Лучшая CV accuracy: 0.845


Вывод

После настройки CV accuracy Random Forest выросла с `0.815` до `0.845`. Лучший результат получен при `max_depth=8`, `min_samples_leaf=1` и `max_features=None`.

У модели остаётся заметный разрыв между train и CV (`0.937` против `0.845`), поэтому лес частично переобучается. Однако по CV accuracy он становится текущим лидером.

Преимущество над SVC составляет только `0.010`, поэтому пока не считаем Random Forest однозначным победителем.

#### Эксперимент 5: CatBoost

CatBoost — градиентный бустинг с нативной поддержкой категориальных признаков. Поэтому для него не используем `OneHotEncoder` и `StandardScaler`: после общего feature engineering категории передаются модели напрямую.

Сначала проверим стабильную конфигурацию. Метрику и CV-фолды оставляем прежними, чтобы результат можно было сравнить с остальными моделями. Ниже сохранён использованный поиск параметров.

In [4]:
# Загружаем стабильную конфигурацию CatBoost.
config = OmegaConf.load(CONFIG_DIR/'catboost'/'01_stable.yaml')

# Создаём CatBoostClassifier из YAML-конфигурации.
model = instantiate(config.model)

# Собираем CatBoost pipeline.
pipeline = build_catboost_pipeline(model)

# Параметр адресуется конкретному шагу pipeline.
catboost_fit_params = {
    'catboostclassifier__cat_features': list(
        config.fit.cat_features
    )
}

# Оцениваем CatBoost на общей схеме кросс-валидации.
catboost_scores = evaluate(
    pipeline=pipeline,
    X=X,
    y=y,
    fit_params=catboost_fit_params
)

Accuracy по фолдам: 0.844, 0.854, 0.826, 0.831, 0.854
Средняя accuracy: 0.842
Std accuracy: 0.011


Посмотрим итоговое качество стабильного CatBoost на кросс-валидации.

In [5]:
catboost_result = (
    pd.Series({
        'Модель': 'CatBoost',
        'CV accuracy': catboost_scores.mean(),
        'CV std': catboost_scores.std(),
    })
    .to_frame()
    .T
)

display(
    catboost_result.style
    .hide(axis='index')
    .format({
        'CV accuracy': '{:.3f}',
        'CV std': '{:.3f}',
    })
)

Модель,CV accuracy,CV std
CatBoost,0.842,0.011


Вывод

Стабильный CatBoost получил CV accuracy `0.842` при `std = 0.011`. Он превосходит SVC на `0.007`.

До Random Forest модели не хватает `0.003`, поэтому по текущей CV они фактически идут рядом.

**Итерация 2: подбор основных параметров CatBoost**

Проверим количество деревьев, `learning_rate`, глубину и L2-регуляризацию.

`iterations` и `learning_rate` определяют темп обучения ансамбля. `depth` управляет сложностью отдельных деревьев, а `l2_leaf_reg` ограничивает значения в листьях и помогает бороться с переобучением.

Остальные настройки, признаки и CV-фолды не меняем.

In [8]:
# Загружаем конфигурацию CatBoost и сетку поиска.
config = OmegaConf.load(CONFIG_DIR/'catboost'/'02_grid_search.yaml')

# Создаём CatBoostClassifier с параметрами предыдущей итерации.
model = instantiate(config.model)

# Собираем pipeline с нативной
# обработкой категориальных признаков.
pipeline = build_catboost_pipeline(model)

# Передаём GridSearchCV основные параметры CatBoost.
param_grid = {
    'catboostclassifier__iterations': list(
        config.search.iterations
    ),
    'catboostclassifier__learning_rate': list(
        config.search.learning_rate
    ),
    'catboostclassifier__depth': list(
        config.search.depth
    ),
    'catboostclassifier__l2_leaf_reg': list(
        config.search.l2_leaf_reg
    )
}

# Категориальные признаки передаём в CatBoost.fit(),
# а не в конструктор модели.
catboost_fit_params = {
    'catboostclassifier__cat_features': list(
        config.fit.cat_features
    )
}

# Проверяем все комбинации на общей схеме CV.
catboost_search = run_grid_search(
    pipeline=pipeline,
    param_grid=param_grid,
    X=X,
    y=y,
    fit_params=catboost_fit_params
)

Выведем лучшие комбинации параметров. Разница между train и CV покажет, насколько сильно CatBoost подстраивается под обучающие данные.

In [9]:
result_columns = {
    'rank_test_score': 'Rank',
    'param_catboostclassifier__iterations': 'Iterations',
    'param_catboostclassifier__learning_rate': 'Learning rate',
    'param_catboostclassifier__depth': 'Depth',
    'param_catboostclassifier__l2_leaf_reg': 'L2 leaf reg',
    'mean_train_score': 'Train accuracy',
    'mean_test_score': 'CV accuracy',
    'std_test_score': 'CV std'
}

catboost_search_results = (
    pd.DataFrame(catboost_search.cv_results_)
    .rename(columns=result_columns)
    .filter(items=tuple(result_columns.values()))
    .astype({
        'Rank': 'int64',
        'Iterations': 'int64',
        'Learning rate': 'float64',
        'Depth': 'int64',
        'L2 leaf reg': 'float64'
    })
    .sort_values(['Rank', 'CV std'])
    .reset_index(drop=True)
)

display(
    catboost_search_results
    .head(15)
    .style
    .hide(axis='index')
    .format({
        'Learning rate': '{:.2f}',
        'L2 leaf reg': '{:g}',
        'Train accuracy': '{:.3f}',
        'CV accuracy': '{:.3f}',
        'CV std': '{:.3f}'
    })
)

print('Лучшее количество деревьев:', catboost_search.best_params_['catboostclassifier__iterations'])
print('Лучший learning_rate:', catboost_search.best_params_['catboostclassifier__learning_rate'])
print('Лучшая глубина:', catboost_search.best_params_['catboostclassifier__depth'])
print('Лучший l2_leaf_reg:', catboost_search.best_params_['catboostclassifier__l2_leaf_reg'])
print(f'Лучшая CV accuracy: ' f'{catboost_search.best_score_:.3f}')

Rank,Iterations,Learning rate,Depth,L2 leaf reg,Train accuracy,CV accuracy,CV std
1,400,0.03,3,0.5,0.859,0.842,0.011
2,300,0.05,4,1,0.872,0.842,0.025
3,200,0.05,5,1,0.870,0.842,0.030
4,300,0.05,3,0.5,0.862,0.841,0.020
5,300,0.03,4,0.1,0.861,0.841,0.023
6,300,0.05,3,1,0.862,0.840,0.018
7,400,0.05,4,1,0.884,0.839,0.025
8,300,0.03,5,0.5,0.865,0.839,0.017
9,200,0.03,3,0.5,0.842,0.838,0.004
10,200,0.05,4,1,0.861,0.838,0.023


Лучшее количество деревьев: 400
Лучший learning_rate: 0.03
Лучшая глубина: 3
Лучший l2_leaf_reg: 0.5
Лучшая CV accuracy: 0.842


Вывод

Настройка подняла CV accuracy CatBoost с `0.835` до `0.842`. Лучшие параметры: `iterations=400`, `learning_rate=0.03`, `depth=3`, `l2_leaf_reg=0.5`.

Разрыв между train (`0.859`) и CV (`0.842`) небольшой, а `CV std=0.011` — результат достаточно стабильный. Дальше CatBoost тюнить не будем: улучшения остановились.

#### Эксперимент 6: Fully Connected Neural Network

Проверим небольшую полносвязную сеть на том же наборе признаков. После feature engineering категориальные признаки кодируются через `OneHotEncoder`, а числовые признаки заполняются и масштабируются.

Сеть оставляем небольшой: на 891 наблюдении сложная архитектура быстро начнёт запоминать обучающие данные.

In [3]:
# Загружаем стабильную конфигурацию нейронной сети.
config = OmegaConf.load(CONFIG_DIR/'neural_network'/'01_stable.yaml')

# Фиксируем инициализацию весов, dropout
# и перемешивание обучающих батчей.
torch.manual_seed(
    int(config.runtime.random_seed)
)

# Hydra создаёт sklearn-совместимую обёртку
# над нашей PyTorch-моделью.
model = instantiate(config.model)

# Собираем полный pipeline:
# feature engineering -> preprocessing -> FCNN.
pipeline = build_neural_pipeline(model)

# Обучаем фолды последовательно, чтобы несколько
# экземпляров PyTorch не запускались одновременно.
fcnn_scores = evaluate(
    pipeline=pipeline,
    X=X,
    y=y,
    n_jobs=int(config.runtime.n_jobs),
)

Accuracy по фолдам: 0.872, 0.815, 0.792, 0.843, 0.815
Средняя accuracy: 0.827
Std accuracy: 0.027


In [4]:
fcnn_result = (
    pd.Series({
        'Модель': 'FCNN',
        'CV accuracy': fcnn_scores.mean(),
        'CV std': fcnn_scores.std(),
    })
    .to_frame()
    .T
)

display(
    fcnn_result.style
    .hide(axis='index')
    .format({
        'CV accuracy': '{:.3f}',
        'CV std': '{:.3f}',
    })
)

Модель,CV accuracy,CV std
FCNN,0.827,0.027


Вывод

FCNN получила CV accuracy `0.827` при `std = 0.027`. Она уступает классическим моделям и заметно сильнее зависит от конкретного разбиения данных.

Для нейронной сети выборка из 891 пассажира слишком мала: преимуществ сложной модели здесь использовать негде, а риск переобучения высокий. Поэтому FCNN оставляем как учебный эксперимент и дальше не настраиваем.

#### Эксперимент 7: Hard Voting

Объединим стабильные Logistic Regression, SVC и Random Forest. Каждая модель предсказывает класс пассажира, после чего ансамбль выбирает ответ большинством голосов.

В ансамбль берём разные по устройству модели. Decision Tree не используем из-за слабого результата, а CatBoost оставляем отдельно, поскольку ему требуется другой preprocessing.

In [7]:
# Загружаем стабильные конфигурации базовых моделей.
logistic_config = OmegaConf.load(CONFIG_DIR/'logistic_regression'/'01_stable.yaml')
svc_config = OmegaConf.load(CONFIG_DIR/'svc'/'01_stable.yaml')
random_forest_config = OmegaConf.load(CONFIG_DIR/'random_forest'/'01_stable.yaml')
ensemble_config = OmegaConf.load(CONFIG_DIR/'ensemble'/'01_stable.yaml')

# Создаём уже настроенные модели.
logistic_model = instantiate(logistic_config.model)
svc_model = instantiate(svc_config.model)
random_forest_model = instantiate(random_forest_config.model)

# Внешняя CV уже распараллеливает фолды.
# Поэтому Random Forest внутри каждого фолда
# использует только один поток.
random_forest_model.set_params(n_jobs=1)

# Передаём готовые модели в VotingClassifier.
model = instantiate(
    ensemble_config.model,
    estimators=[
        ('logistic_regression', logistic_model),
        ('svc', svc_model),
        ('random_forest', random_forest_model),
    ],
)

# Все три модели используют одинаковые признаки
# и общий sklearn preprocessing.
pipeline = build_pipeline(model)

voting_scores = evaluate(
    pipeline=pipeline,
    X=X,
    y=y,
)

Accuracy по фолдам: 0.849, 0.831, 0.826, 0.843, 0.848
Средняя accuracy: 0.839
Std accuracy: 0.009


In [8]:
voting_result = (
    pd.Series({
        'Модель': 'Hard Voting',
        'CV accuracy': voting_scores.mean(),
        'CV std': voting_scores.std(),
    })
    .to_frame()
    .T
)

display(
    voting_result.style
    .hide(axis='index')
    .format({
        'CV accuracy': '{:.3f}',
        'CV std': '{:.3f}',
    })
)

Модель,CV accuracy,CV std
Hard Voting,0.839,0.009


Вывод

Hard Voting получил CV accuracy `0.839` при `std = 0.009`. Ансамбль работает стабильнее Random Forest, но уступает ему по средней accuracy на `0.006`.

Близкие результаты базовых моделей не дали прироста: Logistic Regression и SVC во многом совершают одинаковые ошибки и формируют большинство голосов. Поэтому Hard Voting не используем, а итоговой моделью оставляем Random Forest.